In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from bertopic import BERTopic
from src.normalisation import normalize

# Charger le corpus complet (train + val) pour avoir assez de docs
train_df = pd.read_csv('../dataset_nlp/splits/train.csv')
val_df = pd.read_csv('../dataset_nlp/splits/val.csv')
corpus = pd.concat([train_df, val_df], ignore_index=True)

# Agréger par shelfmark (une doc = une page de manuscrit)
# BERTopic fonctionne mieux sur des textes un peu longs
docs_by_shelfmark = (
    corpus.groupby('shelfmark')['text']
    .apply(lambda lines: ' '.join(str(l) for l in lines))
    .reset_index()
)
docs_by_shelfmark['text_normalized'] = docs_by_shelfmark['text'].apply(normalize)

print(f"Corpus : {len(corpus)} lignes → {len(docs_by_shelfmark)} documents (par shelfmark)")
print(f"Taille moyenne d'un doc : {docs_by_shelfmark['text'].str.len().mean():.0f} chars")
print(f"\nPremier doc (extrait) : {docs_by_shelfmark['text_normalized'].iloc[0][:200]}")

/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Corpus : 29398 lignes → 34 documents (par shelfmark)
Taille moyenne d'un doc : 48192 chars

Premier doc (extrait) : Qant el vit sõ escu ꝑcie Lo cheual par laresne tint .iii. ch̾rs. co menacoient Q̃ ꝑ sa borde la decoit Gaitant lo võt de ml̾t se plaĩt De sergenz armez .iii. o .iiii. El bois que ele ira ap̾s Li ch


In [2]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Modèle multilingue adapté au français
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# BERTopic avec paramètres adaptés à un petit corpus (34 docs)
topic_model = BERTopic(
    embedding_model=embedding_model,
    language="french",
    min_topic_size=2,       # petit corpus → topics petits acceptés
    nr_topics="auto",
    verbose=True
)

docs = docs_by_shelfmark['text_normalized'].tolist()

print(f"Entraînement BERTopic sur {len(docs)} documents...")
topics, probs = topic_model.fit_transform(docs)

print(f"\nNombre de topics trouvés : {len(set(topics)) - 1} (+ outliers topic -1)")
print(f"\nTop topics :")
print(topic_model.get_topic_info().head(10).to_string())

Loading weights: 100%|███████████████████████| 199/199 [00:00<00:00, 815.32it/s]
2026-06-18 13:10:47,351 - BERTopic - Embedding - Transforming documents to embeddings.


Entraînement BERTopic sur 34 documents...


Batches: 100%|████████████████████████████████████| 2/2 [00:04<00:00,  2.21s/it]
2026-06-18 13:10:51,819 - BERTopic - Embedding - Completed ✓
2026-06-18 13:10:51,822 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-18 13:11:17,610 - BERTopic - Dimensionality - Completed ✓
2026-06-18 13:11:17,616 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-18 13:11:17,648 - BERTopic - Cluster - Completed ✓
2026-06-18 13:11:17,650 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-06-18 13:11:18,140 - BERTopic - Representation - Completed ✓
2026-06-18 13:11:18,142 - BERTopic - Topic reduction - Reducing number of topics
2026-06-18 13:11:18,160 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-18 13:11:18,563 - BERTopic - Representation - Completed ✓
2026-06-18 13:11:18,569 - BERTopic - Topic reduction - Reduced number of topics from 2 to 2



Nombre de topics trouvés : 1 (+ outliers topic -1)

Top topics :
   Topic  Count              Name                                Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [3]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# Paramètres adaptés aux petits corpus (34 docs)
umap_model = UMAP(
    n_neighbors=5,       # réduit car peu de docs (défaut=15)
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=2,  # cluster de 2 docs minimum (défaut=10)
    min_samples=1,
    metric='euclidean',
    prediction_data=True
)

topic_model_v2 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="french",
    min_topic_size=2,
    nr_topics="auto",
    verbose=True
)

docs = docs_by_shelfmark['text_normalized'].tolist()

print(f"Entraînement BERTopic v2 sur {len(docs)} documents...")
topics_v2, probs_v2 = topic_model_v2.fit_transform(docs)

print(f"\nNombre de topics trouvés : {len(set(topics_v2)) - 1} (+ outliers topic -1)")
print(f"\nTop topics :")
print(topic_model_v2.get_topic_info().to_string())

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 1060.79it/s]
2026-06-18 13:29:35,320 - BERTopic - Embedding - Transforming documents to embeddings.


Entraînement BERTopic v2 sur 34 documents...


Batches: 100%|████████████████████████████████████| 2/2 [00:01<00:00,  1.32it/s]
2026-06-18 13:29:36,850 - BERTopic - Embedding - Completed ✓
2026-06-18 13:29:36,851 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-18 13:29:37,531 - BERTopic - Dimensionality - Completed ✓
2026-06-18 13:29:37,532 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-18 13:29:37,543 - BERTopic - Cluster - Completed ✓
2026-06-18 13:29:37,544 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-06-18 13:29:37,775 - BERTopic - Representation - Completed ✓
2026-06-18 13:29:37,776 - BERTopic - Topic reduction - Reducing number of topics
2026-06-18 13:29:37,783 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-18 13:29:38,007 - BERTopic - Representation - Completed ✓
2026-06-18 13:29:38,009 - BERTopic - Topic reduction - Reduced number of topics from 2 to 2



Nombre de topics trouvés : 1 (+ outliers topic -1)

Top topics :
   Topic  Count              Name                                Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [4]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import KMeans

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

umap_model = UMAP(
    n_neighbors=5,
    n_components=3,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# KMeans force 5 clusters au lieu de HDBSCAN
from sklearn.cluster import KMeans
cluster_model = KMeans(n_clusters=5, random_state=42)

representation_model = KeyBERTInspired()

topic_model_v3 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    representation_model=representation_model,
    verbose=True
)

docs = docs_by_shelfmark['text_normalized'].tolist()

print(f"Entraînement BERTopic v3 (KMeans forcé, 5 topics)...")
topics_v3, probs_v3 = topic_model_v3.fit_transform(docs)

print(f"\nTopics trouvés : {len(set(topics_v3))}")
topic_info = topic_model_v3.get_topic_info()
print(topic_info[['Topic', 'Count', 'Name']].to_string())

print("\n--- Mots-clés par topic ---")
for topic_id in sorted(set(topics_v3)):
    words = topic_model_v3.get_topic(topic_id)
    if words:
        print(f"Topic {topic_id}: {[w for w, _ in words[:8]]}")

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 1077.58it/s]
2026-06-18 13:32:33,265 - BERTopic - Embedding - Transforming documents to embeddings.


Entraînement BERTopic v3 (KMeans forcé, 5 topics)...


Batches: 100%|████████████████████████████████████| 2/2 [00:01<00:00,  1.35it/s]
2026-06-18 13:32:34,766 - BERTopic - Embedding - Completed ✓
2026-06-18 13:32:34,767 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-18 13:32:34,816 - BERTopic - Dimensionality - Completed ✓
2026-06-18 13:32:34,817 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-18 13:32:34,991 - BERTopic - Cluster - Completed ✓
2026-06-18 13:32:34,999 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-18 13:32:36,190 - BERTopic - Representation - Completed ✓



Topics trouvés : 5
   Topic  Count                       Name
0      0     11        0_leur_les_estre_le
1      1      7  1_leurs_leur_estoient_les
2      2      6     2_leur_auoir_estre_les
3      3      6    3_leur_les_ville_estoit
4      4      4     4_leur_auoir_estre_les

--- Mots-clés par topic ---
Topic 0: ['leur', 'les', 'estre', 'le', 'vos', 'au', 'auoit', 'qui']
Topic 1: ['leurs', 'leur', 'estoient', 'les', 'estre', 'estoit', 'este', 'avec']
Topic 2: ['leur', 'auoir', 'estre', 'les', 'aux', 'estoit', 'auoit', 'le']
Topic 3: ['leur', 'les', 'ville', 'estoit', 'le', 'leurs', 'estoient', 'furent']
Topic 4: ['leur', 'auoir', 'estre', 'les', 'auoit', 'le', 'au', 'vous']


In [5]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

umap_model = UMAP(
    n_neighbors=5,
    n_components=3,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

cluster_model = KMeans(n_clusters=5, random_state=42)
representation_model = KeyBERTInspired()

# Stopwords moyen français : grammaticaux + très fréquents
stopwords_mfr = [
    # Articles et déterminants
    "le", "la", "les", "un", "une", "des", "du", "au", "aux",
    "ledit", "ladite", "lesdits", "lesdites", "dudit", "audit",
    # Pronoms
    "il", "ils", "elle", "elles", "ce", "se", "si", "en", "y",
    "qui", "que", "qu", "quil", "quelle", "dont", "ou", "ou",
    "lui", "leur", "leurs", "eux", "nous", "vous", "je", "tu",
    # Prépositions et conjonctions
    "et", "de", "a", "en", "par", "pour", "sur", "sous", "sans",
    "avec", "mais", "ou", "car", "ne", "ni", "plus", "bien",
    # Verbes auxiliaires / copule médiévaux
    "estre", "este", "estoit", "estoient", "est", "sont", "sera",
    "avoir", "auoir", "auoit", "auoient", "avoit", "ont", "ont",
    "ait", "soit", "furent", "fut", "feust", "fust",
    "faire", "fait", "faict", "feist",
    # Adverbes fréquents
    "aussi", "ainsi", "encore", "toujours", "jamais", "lors",
    "tant", "tout", "tous", "toute", "toutes", "trop", "tres",
    # Autres très fréquents
    "vos", "ses", "son", "sa", "mon", "ma", "mes", "nos",
    "autre", "autres", "comme", "cõme", "cõ", "dont",
]

vectorizer_model = CountVectorizer(
    stop_words=stopwords_mfr,
    min_df=1,
    ngram_range=(1, 2)
)

topic_model_v4 = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    representation_model=representation_model,
    vectorizer_model=vectorizer_model,
    verbose=True
)

docs = docs_by_shelfmark['text_normalized'].tolist()

print("Entraînement BERTopic v4 (avec stopwords moyen français)...")
topics_v4, probs_v4 = topic_model_v4.fit_transform(docs)

print(f"\nTopics trouvés : {len(set(topics_v4))}")
print("\n--- Mots-clés par topic ---")
for topic_id in sorted(set(topics_v4)):
    words = topic_model_v4.get_topic(topic_id)
    if words:
        print(f"Topic {topic_id}: {[w for w, _ in words[:10]]}")

print("\n--- Distribution des documents ---")
docs_by_shelfmark['topic_v4'] = topics_v4
print(docs_by_shelfmark[['shelfmark', 'topic_v4']].to_string())

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 1646.26it/s]
2026-06-18 13:33:30,724 - BERTopic - Embedding - Transforming documents to embeddings.


Entraînement BERTopic v4 (avec stopwords moyen français)...


Batches: 100%|████████████████████████████████████| 2/2 [00:01<00:00,  1.49it/s]
2026-06-18 13:33:32,085 - BERTopic - Embedding - Completed ✓
2026-06-18 13:33:32,086 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-18 13:33:32,135 - BERTopic - Dimensionality - Completed ✓
2026-06-18 13:33:32,137 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-18 13:33:32,176 - BERTopic - Cluster - Completed ✓
2026-06-18 13:33:32,182 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-18 13:33:33,900 - BERTopic - Representation - Completed ✓



Topics trouvés : 5

--- Mots-clés par topic ---
Topic 0: ['li rois', 'li', 'vi', 'escu', 'liure', 'lor', 'sui', 'dit', 'roi', 'ꝯme']
Topic 1: ['nostredit', 'umble supplicacion', 'seigneur', 'supplicacion', 'ville', 'avecques', 'lesdiz', 'dit', 'tousjours', 'avons']
Topic 2: ['vie', 'iesu crist', 'souuent', 'nostre seigneur', 'euesque', 'engendra', 'dit', 'seigneur', 'laquelle', 'moine']
Topic 3: ['auequez', 'pource', 'ville', 'eust', 'auant', 'voꝰ', 'bertrand', 'li', 'dit', 'ans']
Topic 4: ['vie', 'pource', 'dites', 'ainssi', 'lautre', 'li', 'ꝯme', 'dit', 'quel', 'mie']

--- Distribution des documents ---
                                                 shelfmark  topic_v4
0                              Bern, Burgerbibliothek, 354         0
1                                     Bruxelles, KBR, 9232         2
2                                         Paris, AN, JJ207         1
3                                         Paris, AN, JJ210         1
4                                Paris, B

In [6]:
import json

results = {
    "model": "BERTopic v4 - KMeans(5) + stopwords moyen français",
    "n_docs": len(docs),
    "n_topics": 5,
    "topics": {}
}

topic_labels = {
    0: "Littérature courtoise / récits chevaleresques",
    1: "Actes royaux / registres de chancellerie",
    2: "Textes hagiographiques / religieux",
    3: "Chroniques / récits historiques",
    4: "Textes moraux / didactiques"
}

for topic_id in sorted(set(topics_v4)):
    words = topic_model_v4.get_topic(topic_id)
    docs_in_topic = docs_by_shelfmark[docs_by_shelfmark['topic_v4'] == topic_id]['shelfmark'].tolist()
    results["topics"][str(topic_id)] = {
        "label": topic_labels[topic_id],
        "keywords": [w for w, _ in words[:10]],
        "documents": docs_in_topic,
        "count": len(docs_in_topic)
    }

print(json.dumps(results, ensure_ascii=False, indent=2))

{
  "model": "BERTopic v4 - KMeans(5) + stopwords moyen français",
  "n_docs": 34,
  "n_topics": 5,
  "topics": {
    "0": {
      "label": "Littérature courtoise / récits chevaleresques",
      "keywords": [
        "li rois",
        "li",
        "vi",
        "escu",
        "liure",
        "lor",
        "sui",
        "dit",
        "roi",
        "ꝯme"
      ],
      "documents": [
        "Bern, Burgerbibliothek, 354",
        "Paris, BnF, Arsenal, 3525",
        "Paris, BnF, Velins 488",
        "Paris, BnF, Velins 611",
        "Paris, BnF, fr. 11610",
        "Paris, BnF, fr. 12551",
        "Paris, BnF, fr. 12779",
        "Paris, BnF, fr. 146",
        "Paris, BnF, fr. 1728",
        "Paris, BnF, fr. 411",
        "Vatican, Biblioteca Apostolica Vaticana, Reg.lat. 1616"
      ],
      "count": 11
    },
    "1": {
      "label": "Actes royaux / registres de chancellerie",
      "keywords": [
        "nostredit",
        "umble supplicacion",
        "seigneur",
        "s